# 빅데이터 분석기사 실기 제3유형 학습 노트



## 1. 정규성 및 등분산 검정




### 정규성 검정

In [ ]:
# 정규성 검정
from scipy.stats import shapiro
stat1, p1 = shapiro(group1)
stat2, p2 = shapiro(group2)



```
# 정규성 만족 → t-test
# 정규성 불만족 → wilcoxon(대응표본) / mannwhitneyu(독립표본) / kruskal(3집단)
```



### Kolmogorov-Smirnov 검정 (K-S 검정)

In [ ]:
# K-S 검정: 두 표본이 동일한 분포를 따르는지, 동일한 모집단에서 추출되었는지에 대한 검정
from scipy.stats import ks_2samp
statistic, p_value = ks_2samp(group1, group2)
print(statistic, p_value)

### 등분산 검정 (Levene)

In [ ]:
# 등분산 검정
from scipy.stats import levene
statistic, p_value = levene(group1, group2)
print(statistic, p_value)

## 2. 모수적 평균 검정 (t-test & ANOVA)

### 일표본 t-test (One-sample t-test)

In [ ]:
# 일표본 t-test
from scipy.stats import ttest_1samp
statistic, p_value = ttest_1samp(data, popmean=100, alternative='two-sided')
print(statistic, p_value)



```
# alternative:
# - 'two-sided': 양측검정 (귀무가설: 모평균 = popmean) - 기본값
# - 'less': 단측검정 (대립가설: 모평균 < popmean)
# - 'greater': 단측검정 (대립가설: 모평균 > popmean)
```



### 대응표본 t-test (Paired t-test)



In [ ]:
# 대응표본 t-test
from scipy.stats import ttest_rel
statistic, p_value = ttest_rel(before, after, alternative='two-sided')
print(statistic, p_value)



```
# alternative:
# - 'two-sided': 양측검정 (귀무가설: before 평균 = after 평균) - 기본값
# - 'less': 단측검정 (대립가설: before 평균 < after 평균)
# - 'greater': 단측검정 (대립가설: before 평균 > after 평균)
```



### 이표본 t-test (Independent two-sample t-test)

In [ ]:
# 분산이 같은 이표본 t-test
from scipy.stats import ttest_ind
statistic, p_value = ttest_ind(group1, group2, equal_var=True, alternative='two-sided')
print(statistic, p_value)

# 분산이 다른 이표본 t-test (Welch's t-test)
statistic, p_value = ttest_ind(group1, group2, equal_var=False, alternative='two-sided')
print(statistic, p_value)



```
# alternative:
# - 'two-sided': 양측검정 (귀무가설: group1 평균 = group2 평균) - 기본값
# - 'less': 단측검정 (대립가설: group1 평균 < group2 평균)
# - 'greater': 단측검정 (대립가설: group1 평균 > group2 평균)
```



### 일원분산분석 (One-way ANOVA)

In [ ]:
# 일원분산분석 (ANOVA)
from scipy.stats import f_oneway
statistic, p_value = f_oneway(group1, group2, group3)
print(statistic, p_value)

### 이원분산분석 (Two-way ANOVA)

In [ ]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm # 그룹 효과 유의성 확인(F검정)

# 1) 교호작용 고려 없이 이원분산분석
# C()는 해당 변수가 범주형(Categorical) 변수임을 명시함
formula = 'value ~ C(factor_a) + C(factor_b)'
model = ols(formula, data=df).fit()
anova_table = anova_lm(model)
print(anova_table)

# 2) 교호작용 고려한 이원분산분석 (*은 두 요인의 주효과와 교호작용을 모두 포함)
formula = 'value ~ C(factor_a) * C(factor_b)'
model = ols(formula, data=df).fit()
anova_table = anova_lm(model)
print(anova_table)

## 3. 비모수적 가설 검정 (정규성 불만족 시)

### 부호순위검정 (Wilcoxon Signed-Rank Test)

In [ ]:
# 부호순위검정 (대응표본 비모수)
from scipy.stats import wilcoxon
statistic, p_value = wilcoxon(before, after)


### Mann-Whitney U 검정 (독립표본 비모수)


In [ ]:
# 정규성 불만족하는 두 개의 독립표본 비교
from scipy.stats import mannwhitneyu
statistic, p_value = mannwhitneyu(group1, group2)

### Kruskal-Wallis 검정 (3집단 이상 비모수)

In [ ]:
# 정규성 불만족하는 세 집단 이상 비교
from scipy.stats import kruskal
statistic, p_value = kruskal(group1, group2, group3)

## 4. 범주형 데이터 분석 (카이제곱 검정)

### 카이제곱 독립성 / 동질성 검정

In [ ]:
# 카이제곱 독립성 / 동질성 검정
from scipy.stats import chi2_contingency
table = pd.crosstab(df['a_col'], df['b_col'])
statistic, p_value, dof, expected = chi2_contingency(table)



### 피셔의 직접 검정 (Fisher's Exact Test)

In [ ]:
# 2x2 분할표에서 기대 빈도가 5 미만인 셀이 많아 카이제곱 검정이 부적절할 때 사용
from scipy.stats import fisher_exact
oddsratio, p_value = fisher_exact(table)



### 카이제곱 적합도 검정 (Goodness of Fit)

In [ ]:
# 관측 빈도가 특정 기대 빈도 분포와 부합하는지 검정
from scipy.stats import chisquare
statistic, p_value = chisquare(f_obs, f_exp) # f_obs: 관측 빈도, f_exp: 기대 빈도 (생략 시 균등 분포로 가정)

## 5. 상관관계 및 회귀분석 (선형회귀 / 로지스틱회귀)

### 상관분석

In [ ]:
# 피어슨 상관계수 구하기
from scipy.stats import pearsonr
corr, p = pearsonr(x, y)

# 스피어먼 상관계수 구하기
from scipy.stats import spearmanr
corr, p = spearmanr(x, y)

### 다중회귀분석 (OLS)

In [ ]:
# 다중회귀분석
from statsmodels.formula.api import ols
model = ols('y ~ x1 + x2', data=df).fit()
print(model.summary())

In [ ]:
# 다중회귀분석 상세 통계 지표 추출
r2 = model.rsquared            # 결정계수 (R-squared)
adj_r2 = model.rsquared_adj    # 조정된 결정계수
f_val = model.fvalue           # F-통계량
f_pval = model.f_pvalue        # F-통계량의 p-value
coef = model.params            # 회귀계수 (Series)
p_vals = model.pvalues         # 각 독립변수 회귀계수의 p-value (Series)
std_err = model.bse            # 각 독립변수 회귀계수의 표준오차 (Series)
conf_int = model.conf_int()    # 신뢰구간 (DataFrame)

In [ ]:
# 특정 독립변수 값에 대한 예측값(y) 구하기
# 예시: x1=1.5, x2=2.0 일 때 예측값 계산
import pandas as pd
new_data = pd.DataFrame({'x1': [1.5], 'x2': [2.0]})
predicted_val = model.predict(new_data)
print(predicted_val[0])

### 로지스틱 회귀분석 (Logit / GLM)

In [ ]:
# 1) Logit 모델 사용 방법
from statsmodels.formula.api import logit
model = logit('y ~ x1 + x2', data=df).fit()
print(model.summary())

# 로지스틱회귀분석 상세 통계 지표 추출
pseudo_r2 = model.prsquared     # 의사 결정계수 (McFadden's R2)
aic = model.aic                 # AIC
bic = model.bic                 # BIC
coef = model.params             # 회귀계수 (Series)
p_vals = model.pvalues          # p-value (Series)
deviance = -2 * model.llf       # 잔차 이탈도
null_deviance = -2 * model.llnull

# 오즈비 구하기
import numpy as np
odds_ratio = np.exp(model.params)

# 이탈도(Deviance) 계산: LogitResults 객체에는 deviance 속성이 없음
deviance = -2 * model.llf
null_deviance = -2 * model.llnull

In [ ]:
# 2) GLM을 사용하여 로지스틱 회귀 수행 (Deviance를 바로 호출해야 하는 경우 편리)
from statsmodels.formula.api import glm
import statsmodels.api as sm
model_glm = glm('y ~ x1 + x2', data=df, family=sm.families.Binomial()).fit()
deviance = model_glm.deviance
null_deviance = model_glm.null_deviance

# 특정 독립변수 값에 대한 예측 확률 구하기 (y=1일 확률)
# 예시: x1=1.5, x2=2.0 일 때 성공 확률 계산
new_data = pd.DataFrame({'x1': [1.5], 'x2': [2.0]})
predicted_prob = model.predict(new_data) # 확률값(0~1) 반환
print(predicted_prob[0])